# Check 1: Are village roads and hospitals mapped?

**Why this matters:** the whole "cut off" analysis depends on OpenStreetMap (OSM) having the small village roads. If only highways are mapped, every village will look cut off and the results mean nothing. This is the biggest risk in the project, so we check it first.

**What this notebook does**
1. Downloads the OSM road network and health facilities for Sivasagar, Charaideo and Jorhat.
2. Measures how much of the network is village roads versus highways.
3. Shows a satellite map with OSM roads on top, so you can see with your own eyes whether village roads are missing.

**Run it in Google Colab** (it needs open internet). It takes about 5 to 15 minutes, mostly downloading.

In [ ]:
!pip install -q osmnx folium

In [ ]:
# Get the project code (config, preprocessing, metrics) from GitHub.
# Change REPO_URL if your repository lives somewhere else.
import os, sys, subprocess
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from src import config
print("Project code loaded from", REPO)

In [ ]:
import datetime
import osmnx as ox
import pandas as pd

TODAY = datetime.date.today().isoformat()
RAW = f"{REPO}/data/raw"
RESULTS = f"{REPO}/results"
os.makedirs(RAW, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

ox.settings.use_cache = True
ox.settings.requests_timeout = 300

# To test the backup area instead, use: DISTRICTS = config.BACKUP_DISTRICT
DISTRICTS = config.DISTRICTS

import time
def with_retry(fn, *args, tries=5, wait=30, **kwargs):
    """The public OSM server is often busy. Retry with growing waits instead of giving up."""
    for attempt in range(1, tries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == tries:
                raise
            print(f"  server busy or error ({str(e)[:80]}...). Waiting {wait}s, attempt {attempt}/{tries}")
            time.sleep(wait)
            wait *= 2

In [ ]:
def road_class(h):
    """OSM 'highway' can be a list; keep the first value."""
    return h[0] if isinstance(h, list) else h


def check_district(name, info):
    print(f"\n=== {name} ===")
    # 1. District boundary (to measure area)
    # Retry the proper district boundary several times. The rough box overlaps
    # neighbouring districts, so its numbers are NOT comparable; use it only as a last resort.
    try:
        boundary = with_retry(ox.geocode_to_gdf, info["place"])
        G = with_retry(ox.graph_from_place, info["place"], network_type="all", retain_all=True)
        how = "district boundary"
    except Exception as e:
        print("WARNING: district boundary failed after retries; using the rough box. "
              "These numbers overlap other districts. Rerun this district later.", e)
        boundary = None
        G = with_retry(ox.graph_from_bbox, bbox=info["bbox"], network_type="all", retain_all=True)
        how = "rough bounding box (NOT comparable, rerun)"

    # Save a dated copy of the raw road network (for reproducibility)
    ox.save_graphml(G, f"{RAW}/roads_{name}_{TODAY}.graphml")

    edges = ox.graph_to_gdfs(G, nodes=False)
    edges["cls"] = edges["highway"].apply(road_class)
    edges_m = edges.to_crs(edges.estimate_utm_crs())
    edges["km"] = edges_m.length / 1000

    km_by_class = edges.groupby("cls")["km"].sum().sort_values(ascending=False)
    total_km = km_by_class.sum()
    minor_km = km_by_class[km_by_class.index.isin(config.MINOR_ROAD_TYPES)].sum()
    major_km = km_by_class[km_by_class.index.isin(config.MAJOR_ROAD_TYPES)].sum()

    if boundary is not None:
        area_km2 = boundary.to_crs(boundary.estimate_utm_crs()).area.sum() / 1e6
    else:
        area_km2 = float("nan")

    # 2. Health facilities
    tags = {"amenity": ["hospital", "clinic", "doctors"], "healthcare": True}
    try:
        if boundary is not None:
            health = with_retry(ox.features_from_place, info["place"], tags=tags)
        else:
            health = with_retry(ox.features_from_bbox, bbox=info["bbox"], tags=tags)
        health_out = health.reset_index()[["geometry"] + [c for c in ["name", "amenity", "healthcare"] if c in health.columns]]
        health_out.to_file(f"{RAW}/health_{name}_{TODAY}.geojson", driver="GeoJSON")
        n_health = len(health)
    except Exception as e:
        print("No health facilities found or download failed:", e)
        n_health = 0

    print(f"Area used: {how}")
    print("Road length by type (km), top 12:")
    print(km_by_class.head(12).round(1).to_string())

    return {
        "district": name,
        "area_used": how,
        "area_km2": round(area_km2, 0),
        "road_km_total": round(total_km, 0),
        "road_density_km_per_km2": round(total_km / area_km2, 2) if area_km2 == area_km2 else None,
        "minor_road_share": round(minor_km / total_km, 2) if total_km else 0,
        "major_road_share": round(major_km / total_km, 2) if total_km else 0,
        "health_facilities": n_health,
    }


# To rerun only some districts: rows = [check_district("Jorhat", DISTRICTS["Jorhat"])]
rows = []
for n, i in DISTRICTS.items():
    rows.append(check_district(n, i))
    time.sleep(20)   # be gentle with the public server between districts
summary = pd.DataFrame(rows)
summary.to_csv(f"{RESULTS}/road_check_{TODAY}.csv", index=False)
summary

## How to read the numbers

These are rough guides, not strict rules:

| Measure | Looks good | Warning sign |
|---|---|---|
| `minor_road_share` | above 0.6 (most road length is village roads) | below 0.4 (mostly highways) |
| `road_density_km_per_km2` | above about 1 | well below 0.5 |
| `health_facilities` | 10 or more per district | fewer than 5 |

The numbers alone are not enough. Do the visual check below.

## Visual check: OSM roads on top of satellite imagery

Yellow = village and minor roads in OSM. Red = main roads. Blue markers = health facilities.

**Important:** roads are only downloaded within `RADIUS_M` (3 km) of `CENTER`. Blank areas beyond that circle are expected, not missing data.

**Towns are almost always mapped well, so do not test them.** Set `CENTER` to rural villages 10 to 20 km from the towns, where people actually get cut off. Good choices are areas reported as hit in July 2026: Nazira, Gaurisagar, Amguri, Demow (Sivasagar) and villages in Charaideo. Right-click a village in Google Maps to copy its coordinates.

At each village, look at the satellite image: small roads and tracks with **no yellow line on top** are missing from OSM. Record how many of 4 villages look complete in `NOTES.md`.

In [ ]:
import folium

# Starting points (lat, lon): district towns. Pick villages nearby by panning.
for n, i in DISTRICTS.items():
    print(n, i["town"])

# Paste a RURAL village's (lat, lon) from Google Maps here, not a town.
CENTER = DISTRICTS["Sivasagar"]["town"]   # change this
RADIUS_M = 3000

Gs = with_retry(ox.graph_from_point, CENTER, dist=RADIUS_M, network_type="all", retain_all=True)
e = ox.graph_to_gdfs(Gs, nodes=False).reset_index()
e["cls"] = e["highway"].apply(road_class)
e = e[["cls", "geometry"]]

m = folium.Map(location=CENTER, zoom_start=15, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite").add_to(m)
folium.GeoJson(
    e[e["cls"].isin(config.MAJOR_ROAD_TYPES)],
    style_function=lambda x: {"color": "red", "weight": 3}, name="Main roads").add_to(m)
folium.GeoJson(
    e[~e["cls"].isin(config.MAJOR_ROAD_TYPES)],
    style_function=lambda x: {"color": "yellow", "weight": 2}, name="Minor roads").add_to(m)

try:
    hs = with_retry(ox.features_from_point, CENTER, tags={"amenity": ["hospital", "clinic", "doctors"], "healthcare": True}, dist=RADIUS_M * 3)
    for _, r in hs.iterrows():
        p = r.geometry.centroid
        folium.Marker([p.y, p.x], popup=str(r.get("name", "health facility"))).add_to(m)
except Exception:
    print("No health facilities near this point")

folium.LayerControl().add_to(m)
m

## Decision

Write the results in `NOTES.md` (the table under "Check 1").

- **Pass:** village roads are mostly present in at least one district. Keep that district (or all three).
- **Partial:** good in one district only. Shrink the study area to that district.
- **Fail:** village roads mostly missing everywhere. Set `DISTRICTS = config.BACKUP_DISTRICT` at the top and run again for Barpeta.